<img src="../assets/agent_design_lens.png" alt="Agent 설계 요소 개념도" style="display: block; width: 100%; max-width: 1200px; margin: 18px auto 24px; border-radius: 12px;">


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Prompt Chatbot (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 system prompt, 모델, 실행 방식을 바꿔 보세요. ReAct Agent는 계산이 필요할 때 도구를 호출해 결과를 답변에 반영합니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>질문을 입력해 답변을 확인합니다.</li>
<li>⚙️ 설정에서 일반 Chat 또는 ReAct Agent를 선택합니다. ReAct Agent에는 <code>15% 할인 뒤 3개 가격은?</code>처럼 계산 질문을 입력해 보세요.</li>
</ol>
  </div>
</div>


In [11]:
from pathlib import Path
import json
from time import perf_counter
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lab_core import run_adaptive_react_agent, run_chat, run_chat_stream, run_react_agent
from src.notebook_ui import build_chat_ui
chat_ui = build_chat_ui(run_chat, chat_stream_runner=run_chat_stream, react_agent_runner=run_react_agent, adaptive_react_agent_runner=run_adaptive_react_agent)
chat_ui.launch(inline=True, share=False, server_name='127.0.0.1', show_error=True, width='100%', height=720)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">메시지 role과 준비</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li><code>system</code>은 모델의 역할, 규칙, 언어, 출력 형식을 정합니다.</li>
<li><code>user</code>는 사용자의 질문 또는 요청입니다.</li>
<li><code>assistant</code>는 이전 모델 응답으로, 대화 history에 사용합니다.</li>
</ul>
<p style="margin: 7px 0;">아래 코드는 루트 <code>.env</code>의 <code>LLM_MODEL</code> 설정을 사용합니다.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 코드로 확인하기</div>


In [12]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lesson_support.llm import chat_text
from src.prompt_model.lesson_support.config import load_settings
from src.prompt_model.lab_core import compare_prompts, run_chat
settings = load_settings()
print(f'LLM model: {settings.llm_model}')

LLM model: gpt-5.6-luna


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 prompt 비교하기</h2>
<p style="margin: 7px 0;">UI에서 본 차이를 <code>compare_prompts()</code>로 확인하는 선택 실습입니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">기본 chat 응답</h3>
<p style="margin: 7px 0;">같은 질문을 기본 조건과 표 형식 조건으로 비교합니다.</p>
  </div>
</div>


In [13]:
question = 'LLM의 기본 아키텍처를 설명해줘.'

comparison_system_prompt = """너는 친절한 선생님이다.
초등학생도 이해할 수 있게 쉬운 한국어로 설명한다.
첫 줄에는 한 문장 요약을 쓴다.
그 아래에 핵심 내용을 bullet 3개로 정리한다."""

# comparison_system_prompt를 직접 바꿔 원하는 제약을 실험하세요.
comparison = compare_prompts(question, settings, system_prompt=comparison_system_prompt)
print(comparison['baseline'])

LLM(대규모 언어 모델)의 기본 아키텍처는 대부분 **Transformer**를 기반으로 합니다. Transformer는 문장 속 토큰 간 관계를 효율적으로 파악하는 신경망 구조입니다.

## 1. 입력 토큰화(Tokenization)

문장을 모델이 처리할 수 있는 작은 단위인 **토큰(token)** 으로 나눕니다.

예:

```text
"나는 학교에 간다"
→ ["나는", "학교", "에", "간다"]
```

토큰은 정수 ID로 변환되어 모델에 입력됩니다.

---

## 2. 임베딩(Embedding)

각 토큰 ID를 고차원 벡터로 변환합니다.

```text
토큰 ID → 벡터
```

이 벡터에는 단어의 의미와 다른 토큰과의 관계가 학습됩니다. 예를 들어 `왕`, `여왕`, `남자`, `여자` 같은 단어의 관계가 벡터 공간에 반영될 수 있습니다.

---

## 3. 위치 정보(Positional Encoding)

Transformer는 기본적으로 토큰의 순서를 직접 알지 못합니다. 따라서 각 토큰의 위치 정보를 추가합니다.

예를 들어 다음 두 문장은 토큰이 같아도 의미가 다릅니다.

```text
개가 사람을 물었다
사람이 개를 물었다
```

위치 인코딩 또는 위치 임베딩을 통해 모델은 토큰 순서를 구분합니다. 최근 모델들은 RoPE(Rotary Positional Embedding) 같은 방식을 많이 사용합니다.

---

## 4. Transformer 블록

LLM은 여러 개의 Transformer 블록을 층층이 쌓은 구조입니다. 각 블록은 주로 다음 요소로 구성됩니다.

### 4.1 Self-Attention

각 토큰이 문장 내 다른 토큰을 얼마나 참고해야 하는지 계산합니다.

예를 들어:

```text
철수는 사과를 먹었다. 그는 맛있다고 했다.
```

모델은 `그는`이 `철수`를 가리킬 가능성이 높다는 관계를 Attention으로 파악할 수 있습니다.

Self-Attention은 각 토큰을 다음 세 가지 벡터로 변환

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">system prompt로 제약하기</h3>
<p style="margin: 7px 0;">제목, 표, bullet처럼 확인하기 쉬운 형식을 지정했을 때 응답이 어떻게 달라지는지 봅니다.</p>
  </div>
</div>


In [14]:
print(comparison['constrained'])

LLM은 글을 숫자로 바꾸고, 문맥을 살펴 다음에 올 말을 예측하는 인공지능입니다.

- **1. 입력을 숫자로 바꾸기**
  - 문장을 작은 단위인 **토큰**으로 나눕니다.
  - 각 토큰을 컴퓨터가 이해할 수 있는 숫자 벡터로 바꿉니다.
  - 예: “나는 밥을 먹었다” → “나는”, “밥”, “을”, “먹었다” 같은 단위로 나눕니다.

- **2. 문맥과 관계 살펴보기**
  - LLM의 중심에는 **트랜스포머(Transformer)**라는 구조가 있습니다.
  - 트랜스포머의 **어텐션(Attention)**은 문장 속 단어들이 서로 어떤 관계가 있는지 살핍니다.
  - 예를 들어 “철수가 사과를 먹었다. 그는 맛있다고 했다.”에서 “그”가 “철수”를 가리킨다는 것을 파악합니다.

- **3. 다음 토큰 예측하기**
  - 처리한 정보를 바탕으로 다음에 올 단어 또는 토큰의 가능성을 계산합니다.
  - 가장 알맞은 토큰을 하나씩 골라 문장을 만듭니다.
  - 이 과정을 반복해 질문에 대한 답이나 글을 완성합니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 2</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">message 배열을 직접 구성하기</h2>
<p style="margin: 7px 0;">LLM API에는 역할이 붙은 메시지 배열을 보냅니다. 아래에서 <code>system</code> 규칙, 이전 대화, 새 질문의 순서를 확인해 보세요.</p>
  </div>
</div>


In [15]:
messages = [
    {"role": "system", "content": "너는 친절한 교육 보조자다. 한국어로 간결하게 답한다."},
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
    {"role": "user", "content": "방금 답을 표로 정리해줘."},
]
for index, item in enumerate(messages, start=1):
    print(f'{index}. {item["role"]}: {item["content"]}')

reply = chat_text(settings, messages, settings.llm_model)
print()
print('응답:')
print(reply)

1. system: 너는 친절한 교육 보조자다. 한국어로 간결하게 답한다.
2. user: LLM의 장점을 두 가지 알려줘.
3. assistant: 자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다.
4. user: 방금 답을 표로 정리해줘.

응답:
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 답변할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 요약, 번역, 작성, 교정 등 여러 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">대화 history가 맥락을 전달하는 방식</h2>
<p style="margin: 7px 0;">두 번째 질문만으로는 무엇을 표로 바꿔야 하는지 알기 어렵습니다. history 유무에 따른 결과를 비교해 보세요.</p>
<p style="margin: 7px 0;">최근 12개 메시지, 총 12,000자까지만 history를 유지합니다.</p>
  </div>
</div>


In [16]:
follow_up = '방금 답을 표로 정리해줘.'
history = [
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
]
without_history = run_chat(follow_up, settings)
with_history = run_chat(follow_up, settings, history=history)

print('='*20,'[history 없음]', '='*20)
print(without_history['reply'])
print()
print('='*20, '[history 포함]','='*20)
print(with_history['reply'])

==================== [history 없음] ====================
표로 정리할 내용이 현재 대화에 보이지 않습니다. 방금 답변의 내용을 보내주시면 아래와 같이 표로 정리해드릴게요.

| 항목 | 내용 |
|---|---|
| 핵심 내용 |  |
| 주요 특징 |  |
| 장점 |  |
| 단점/주의사항 |  |
| 결론 |  |

==================== [history 포함] ====================
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 응답할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 번역, 요약, 글쓰기, 정보 정리 등 여러 작업을 수행할 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">프롬프트로 JSON 형식 요청하고 검증하기</h2>
<p style="margin: 7px 0;">비정형 고객 리뷰에서 긍정·부정 의견, 요약, 키워드를 JSON으로 추출한 뒤 <code>json.loads()</code>로 실제 파싱을 검증합니다.</p>
  </div>
</div>


In [17]:
# 1) 고객이 남긴 비정형 리뷰 원문입니다.
customer_review = (
    '배송은 하루 만에 와서 정말 빨랐고 포장도 깔끔했어요. 제품 디자인과 화면은 마음에 듭니다. ' 
    '다만 배터리가 생각보다 빨리 닳고, 앱 연결이 가끔 끊겨서 아쉬워요. ' 
    '다음 업데이트에서 연결 안정성이 좋아지면 친구에게도 추천할 것 같아요.'
)
print('[1. 비정형 고객 리뷰]')
print(customer_review)

# 2) 리뷰에서 긍정·부정 의견과 핵심 정보를 JSON으로 추출합니다.
json_format_prompt = (
    '너는 고객 리뷰를 분석하는 교육 보조자다. 반드시 JSON 객체만 반환한다. ' 
    'Markdown 코드 블록이나 설명을 덧붙이지 않는다. ' 
    '키는 positive(문자열 배열), negative(문자열 배열), summary(문자열), keywords(문자열 배열)만 사용한다.'
)
json_outcome = run_chat(
    f'다음 고객 리뷰를 지정한 JSON 형식으로 분석해줘.\n\n{customer_review}',
    settings,
    system_prompt=json_format_prompt,
)
json_text = json_outcome['reply']
print('\n[2. 리뷰 분석 JSON]')
print(json_text)

# 3) 모델의 반환값을 실제 JSON으로 파싱해 형식을 검증합니다.
try:
    parsed = json.loads(json_text)
    print('\n[3. JSON 파싱 성공]')
    print(parsed)
except json.JSONDecodeError:
    print('\n[3. JSON 파싱 실패] 반환 원문을 확인하세요:')
    print(json_text)

[1. 비정형 고객 리뷰]
배송은 하루 만에 와서 정말 빨랐고 포장도 깔끔했어요. 제품 디자인과 화면은 마음에 듭니다. 다만 배터리가 생각보다 빨리 닳고, 앱 연결이 가끔 끊겨서 아쉬워요. 다음 업데이트에서 연결 안정성이 좋아지면 친구에게도 추천할 것 같아요.

[2. 리뷰 분석 JSON]
{"positive":["배송이 하루 만에 도착해 매우 빨랐음","포장이 깔끔했음","제품 디자인과 화면이 마음에 듦"],"negative":["배터리가 예상보다 빨리 소모됨","앱 연결이 가끔 끊김","연결 안정성 개선이 필요함"],"summary":"배송과 포장, 디자인과 화면에는 만족했지만 배터리 지속 시간과 앱 연결 안정성에는 아쉬움이 있음. 연결 문제가 개선되면 추천할 의향이 있음.","keywords":["빠른 배송","깔끔한 포장","제품 디자인","화면","배터리","앱 연결","연결 안정성","추천 의향"]}

[3. JSON 파싱 성공]
{'positive': ['배송이 하루 만에 도착해 매우 빨랐음', '포장이 깔끔했음', '제품 디자인과 화면이 마음에 듦'], 'negative': ['배터리가 예상보다 빨리 소모됨', '앱 연결이 가끔 끊김', '연결 안정성 개선이 필요함'], 'summary': '배송과 포장, 디자인과 화면에는 만족했지만 배터리 지속 시간과 앱 연결 안정성에는 아쉬움이 있음. 연결 문제가 개선되면 추천할 의향이 있음.', 'keywords': ['빠른 배송', '깔끔한 포장', '제품 디자인', '화면', '배터리', '앱 연결', '연결 안정성', '추천 의향']}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">동일 질문을 모델별로 비교하기</h2>
<p style="margin: 7px 0;">같은 질문으로 모델을 비교해 보세요. 응답 시간, 정확성, 지시 준수, 출력 형식을 함께 확인합니다.</p>
  </div>
</div>


In [18]:
models_to_compare = [settings.llm_model]
# AI Core에 RUNNING 상태로 배포된 모델이 있다면 아래처럼 추가합니다.
models_to_compare.append('gpt-4.1')

comparison_question = 'SAP에서 2026년도에 출시한 제품은?'
for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = run_chat(comparison_question, settings, model=model_name)
    elapsed = perf_counter() - started_at
    print(f'[{model_name}] {elapsed:.1f}초')
    print(outcome['reply'])
    print('='*100 + '\n')

[gpt-5.6-luna] 7.4초
“2026년에 출시한 SAP 제품”은 **신규 제품 출시**와 **기존 제품의 2026년 버전·릴리스**를 어떻게 구분하느냐에 따라 달라집니다. SAP는 연도별 전체 신제품 목록을 하나로 공개하지도 않습니다.

정확히 확인하려면 다음 공식 채널을 기준으로 보는 것이 좋습니다.

- **SAP News Center**: 2026년 발표·출시 뉴스 검색  
- **SAP 제품 페이지 / SAP Store**: 신규 솔루션 및 상용화 여부 확인  
- **SAP Help Portal의 Product Availability Matrix(PAM)**: 릴리스일, 제품 버전, 유지보수 기간 확인  
- 검색어 예시: `site:news.sap.com 2026 launched SAP product`

참고로 **SAP Business Data Cloud, Joule, SAP S/4HANA Cloud, SAP SuccessFactors** 등은 2026년에 새로 출시된 제품이라기보다, 이전에 출시된 제품·서비스가 2026년에 기능 업데이트나 신규 에디션을 받은 경우로 보는 것이 일반적입니다.

원하시면 제가 범위를 나누어 다음 중 하나로 정리해 드릴 수 있습니다.

1. **2026년 신규 출시 제품만**  
2. **2026년에 새 버전이 나온 SAP 제품**  
3. **SAP ERP·AI·데이터·HR 제품별 목록**  
4. **한국에서 판매 또는 출시된 제품만**

[gpt-4.1] 3.1초
현재까지의 정보(2024년 6월 기준)로는 SAP에서 2026년도에 출시 예정이거나 출시된 제품에 대한 공식 발표가 없습니다. SAP는 주로 새로운 클라우드 솔루션, S/4HANA의 업데이트, AI 및 데이터 분석 관련 신기술을 지속적으로 발표하고 있지만, 2026년에 특정 제품이 출시된다는 공식 자료나 뉴스는 아직 공개되지 않았습니다.

SAP의 주요 제품과 서비스는 다음과 같습니다:
- SAP S/4HANA (ERP 솔루션)
- SAP Bu

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🤖  ReAct 코드 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct Agent의 답변 생성 과정</h2>
<p style="margin: 7px 0;">도구 호출부터 최종 답변까지의 흐름을 확인합니다. <code>Thinking</code>에는 다음 행동을 위한 짧은 근거만 표시합니다.</p>
  </div>
</div>


In [19]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from gen_ai_hub.proxy.native.openai import OpenAI
from src.prompt_model.lab_core import _calculator
from src.prompt_model.lesson_support.ai_core import apply_ai_core_environment

react_system_prompt = '''
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
'''.strip()

@tool
def calculate(expression: str) -> str:
    """사칙연산과 괄호가 있는 수식을 계산합니다."""
    return str(_calculator(expression))

apply_ai_core_environment(settings)
model = ChatOpenAI(model=settings.llm_model, api_key="EMPTY", client=OpenAI().chat.completions)
react_agent = create_agent(model=model, tools=[calculate], system_prompt=react_system_prompt)

react_question = '정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?'
max_react_loops = 5  # 최대 도구 실행 횟수; 답이 완성되면 더 일찍 종료
result = react_agent.invoke(
    {"messages": [{"role": "user", "content": react_question}]},
    {"recursion_limit": max_react_loops * 2 + 2},
)

for message in result["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)


Question: 정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?  
Thought: 할인된 단가를 계산한 뒤 3개 가격을 계산합니다.  
Action: calculate  
Action Input: {"expression":"12000*(1-15/100)*3"}
15% 할인된 상품 가격은 10,200원입니다.

3개 구매 시 총액은 **30,600원**입니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🧪 Reasoning Tag 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct 실행 기록을 태그 흐름으로 읽기</h2>
<p style="margin: 7px 0;">질문을 바꿔 실행해 보세요. <code>&lt;thinking&gt;</code>은 다음 행동을 위한 짧은 근거입니다.</p>
<p style="margin: 7px 0;"><code>&lt;thinking&gt;</code>, <code>&lt;evaluation&gt;</code>, <code>&lt;answer&gt;</code>가 어떻게 이어지는지 확인합니다.</p>
  </div>
</div>


In [20]:
import json

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from gen_ai_hub.proxy.native.openai import OpenAI

from src.prompt_model.lab_core import _calculator
from src.prompt_model.lesson_support.ai_core import apply_ai_core_environment


react_system_prompt = '''
너는 Joule 구매·회계·인사·제조원가 지원 ReAct Agent다.
Plan → Thought → Action → Observation → Evaluation → Final Answer 순서를 따른다.

절대로 Question:, Thought:, Action:, Action Input:, Observation:, Final Answer:
같은 텍스트 형식을 사용하지 않는다. XML 태그 형식만 사용한다.

태그 역할:
- <plan>: 해결할 작업의 짧은 계획
- <assumptions>: 답변에 사용한 전제
- <thinking>: 다음 행동을 선택하는 공개 가능한 짧은 근거
- <tool_selection>: 사용할 도구와 선택 이유
- <evaluation>: 도구 결과의 정확성·충분성 점검
- <reflection>: 오류·누락·추가 확인 필요 여부
- <answer>: 최종 사용자 답변

상품, 재고, 예산, 회계 정책, 직원·결재선, 제조원가, 생산능력,
외부 견적, 품질 요건은 반드시 해당 도구로 확인한다.
계산은 calculate 도구로 검증하고, 도구 반환값만 사실로 사용한다.

첫 도구 호출 전에는 다음 태그를 작성한다.

<plan>...</plan>
<assumptions>...</assumptions>
<thinking>...</thinking>
<tool_selection>...</tool_selection>

도구 결과를 받은 뒤 최종 응답은 다음 태그를 작성한다.

<thinking>...</thinking>
<evaluation>...</evaluation>
<reflection>...</reflection>
<answer>...</answer>

각 태그는 한두 문장으로 간결하게 작성한다.
상세 내부 추론이나 장문의 풀이 과정은 작성하지 않는다.
'''.strip()

@tool
def get_facility_staffing_request(site: str) -> str:
    """설비팀: 설비 운영 현황과 필요한 인력 충원 요청을 조회합니다."""
    requests = {
        "PLANT-SEOUL": {
            "request_id": "FAC-REQ-2026-041",
            "site": "PLANT-SEOUL",
            "equipment_area": "조립 라인 설비보전",
            "current_headcount": 4,
            "required_headcount": 6,
            "requested_headcount": 2,
            "position": "설비보전 Technician",
            "reason": "예방정비와 긴급 고장 대응 인력이 부족합니다.",
            "required_start_date": "2026-12-01",
        }
    }
    return json.dumps(
        requests.get(site, {"error": "설비팀 요청을 찾을 수 없습니다."}),
        ensure_ascii=False,
    )


@tool
def prepare_hr_recruitment_request(
    facility_request_id: str,
    position: str,
    headcount: int,
    required_start_date: str,
) -> str:
    """인사팀: 설비팀 인력 요청을 바탕으로 채용요청 초안을 작성합니다."""
    return json.dumps({
        "status": "DRAFT",
        "hr_request_id": "HR-REQ-2026-118",
        "facility_request_id": facility_request_id,
        "position": position,
        "requested_headcount": headcount,
        "required_start_date": required_start_date,
        "estimated_recruiting_days": 30,
        "approval_required": ["설비팀장", "HR Business Partner"],
    }, ensure_ascii=False)


@tool
def get_manufacturing_plan_impact(hr_request_id: str) -> str:
    """제조기획: 채용 요청이 생산계획과 인력 투입 일정에 미치는 영향을 확인합니다."""
    return json.dumps({
        "hr_request_id": hr_request_id,
        "production_plan": "2027년 1분기 조립라인 생산량 20% 증산",
        "recommended_onboarding_date": "2026-12-01",
        "staffing_priority": "HIGH",
        "temporary_workaround": "채용 완료 전까지 외주 정비 인력 1명 투입",
    }, ensure_ascii=False)


@tool
def get_hr_recruitment_sop(request_type: str) -> str:
    """인사팀의 채용 요청 SOP와 필수 검토 항목을 조회합니다."""
    sops = {
        "facility_staffing": {
            "sop_name": "설비 인력 충원 및 채용 요청 SOP",
            "process_sequence": [
                "설비팀 인력 충원 요청 확인",
                "인사팀 채용요청 초안 작성",
                "제조기획팀 생산계획 영향 검토",
                "설비팀장 및 HR Business Partner 결재",
            ],
            "required_fields": [
                "직무",
                "요청 인원",
                "충원 사유",
                "입사 필요일",
                "생산계획 영향",
            ],
            "restriction": "채용 공고 등록 전에는 승인된 채용요청이 필요합니다.",
        }
    }
    return json.dumps(
        sops.get(request_type, {"error": "해당 채용 SOP를 찾을 수 없습니다."}),
        ensure_ascii=False,
    )
    
react_agent = create_agent(
    model=model,
    tools=[
        get_hr_recruitment_sop,
        get_facility_staffing_request,
        prepare_hr_recruitment_request,
        get_manufacturing_plan_impact,
    ],
    system_prompt=react_system_prompt,
)

react_question = '''
PLANT-SEOUL 설비팀에서 설비보전 Technician 인력 충원이 필요합니다.

사내 인사 채용 SOP를 확인해 요청을 검토하고,
채용요청 초안을 준비한 뒤 생산계획 영향까지 확인해 주세요.
실제 채용 공고는 등록하지 않습니다.
'''

max_react_loops = 8

for update in react_agent.stream(
    {"messages": [{"role": "user", "content": react_question}]},
    {"recursion_limit": max_react_loops * 2 + 2},
    stream_mode="updates",
):
    for node_name, state in update.items():
        for message in state.get("messages", []):
            if message.type == "ai" and message.content:
                print(f"\n[{node_name}]")
                print(message.content)

            if message.type == "ai" and message.tool_calls:
                for call in message.tool_calls:
                    print(f"<action>{call['name']}({call['args']})</action>")

            if message.type == "tool":
                print(f"<observation>{message.content}</observation>")


[model]
<plan>채용 SOP와 PLANT-SEOUL 설비팀의 인력 요청을 확인한 후, 확인된 요건으로 채용요청 초안을 작성하고 생산계획 영향을 검토하겠습니다.</plan>
<assumptions>실제 채용 공고 등록은 수행하지 않으며, 도구에서 확인된 정보만 최종 사실로 사용합니다.</assumptions>
<thinking>SOP와 설비팀 요청은 서로 독립적으로 조회할 수 있으므로 병렬 확인이 적절합니다.</thinking>
<tool_selection>인사 채용 SOP 조회와 설비팀 인력 충원 요청 조회를 병렬 실행합니다.</tool_selection>
<action>get_hr_recruitment_sop({'request_type': '설비보전 Technician 인력 충원'})</action>
<action>get_facility_staffing_request({'site': 'PLANT-SEOUL'})</action>
<observation>{"request_id": "FAC-REQ-2026-041", "site": "PLANT-SEOUL", "equipment_area": "조립 라인 설비보전", "current_headcount": 4, "required_headcount": 6, "requested_headcount": 2, "position": "설비보전 Technician", "reason": "예방정비와 긴급 고장 대응 인력이 부족합니다.", "required_start_date": "2026-12-01"}</observation>
<observation>{"error": "해당 채용 SOP를 찾을 수 없습니다."}</observation>

[model]
<thinking>설비팀 요청은 확인되었지만 특정 문구의 SOP 조회가 실패했으므로 일반 채용 요청 기준으로 재조회하겠습니다. 요청 ID와 직무·인원·입사일이 확보되어 초안 작성도 동시에 진행할 수 있습니다.</thinking>
<tool_selection>일반 채용 요청 S